# Week 10: Reproducible ML Workflow Studio

**Monday 9 November 2026, 14:00–17:00 · NCKU + LAiMM**

The goal is a handover another Python process can validate and replay from saved files. Week 9 checked CSV readback in the same kernel; this week starts a new interpreter and working directory. The installed packages still remain shared. This is computational replay of frozen observations, not a new scientific experiment.

## Use one complete notebook

1. Save your own copy (Colab: File → Save a copy in Drive). Choose a free CPU runtime. No GPU, API key, paid AI tool or remote dataset download is needed.
2. Run Setup, then Tasks **1 → 8**. Every task has a prediction cell, runnable scaffold, explanation cell and stopping check. Keep the same notebook for the whole class. The worked alternative is available if you need sample responses; you do not need to run a second copy.
3. H1: Tasks 1–2, 15 practice minutes. H2: Tasks 3–5, 45. H3: Tasks 6–8, 40, then 20 minutes for peer explanation/export/optional work. Total: 60 minutes explanation + 120 practice.
4. A runtime reset removes variables. Restart from Setup and run in order. Setup recreates ONLY this notebook’s `artifacts/week10-reference/handover` output folder; save edited files elsewhere before restarting.
5. Download `week10-workflow-handover.zip` from the Colab Files sidebar (folder icon → artifacts → week10-reference → ZIP menu → Download). In Jupyter use the final file link. Save your edited .ipynb separately.

**Prerequisites:** stable row IDs, train/validation/test, MAE/RMSE, basic Python arrays. The retained scikit-learn packaged diabetes fixture is used only as a statistical regression example: 442 source rows, ten raw numeric inputs. We do not infer medical, causal or deployment validity. Source-row IDs belong to this frozen dataset version and are not global patient identities.

These are in-class formative exercises. Designated Homework and written/oral examinations are unchanged. The next meeting on 16 November is the written and oral midterm examination.


**Student exercise:** write your own predictions and explanations.

## Setup

Run once. Missing free libraries are installed only if absent; existing packages are not upgraded. The cell records actual versions and declares candidates, split sizes and tolerances before fitting.

In [ ]:
import os,sys,subprocess,importlib.util,platform,json,csv,hashlib,zipfile,tempfile,shutil
from pathlib import Path
for k in ['OMP_NUM_THREADS','OPENBLAS_NUM_THREADS','MKL_NUM_THREADS']:os.environ.setdefault(k,'1')
required={'numpy':'numpy','matplotlib':'matplotlib','sklearn':'scikit-learn'}
missing=[p for m,p in required.items() if importlib.util.find_spec(m) is None]
if missing:subprocess.check_call([sys.executable,'-m','pip','install',*missing])
import numpy as np
import matplotlib,matplotlib.pyplot as plt
import sklearn,scipy
from sklearn.datasets import load_diabetes
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from sklearn.dummy import DummyRegressor
from sklearn.metrics import mean_absolute_error,mean_squared_error
CONFIG={'seed':2026,'dataset':'sklearn diabetes raw; statistical teaching only',
 'train_n':265,'validation_n':88,'test_n':89,'alphas':[1.,20.,100.],
 'candidate_order':['median','ridge-1','ridge-20','ridge-100'],
 'selection':'validation MAE; exact tie follows candidate_order','atol':1e-8,'rtol':1e-10}
ENV={'python':platform.python_version(),'numpy':np.__version__,'scikit_learn':sklearn.__version__,'matplotlib':matplotlib.__version__,'scipy':scipy.__version__,'platform':platform.platform()}
RUN_LABEL='reference'
ARTIFACTS=Path('artifacts')/('week10-'+RUN_LABEL);ARTIFACTS.mkdir(parents=True,exist_ok=True)
BUNDLE_DIR=ARTIFACTS/'handover'
if BUNDLE_DIR.exists():shutil.rmtree(BUNDLE_DIR)
BUNDLE_DIR.mkdir()
plt.rcParams.update({'figure.figsize':(10,6.9),'savefig.dpi':160,'font.size':22,
 'axes.labelsize':22,'axes.titlesize':24,'xtick.labelsize':17,'ytick.labelsize':18,
 'legend.fontsize':17,'axes.spines.top':False,'axes.spines.right':False})
TEAL='#167b87';RED='#7f2635';GOLD='#b88a29'
def finish(fig,name):
 fig.savefig(ARTIFACTS/name,bbox_inches='tight',facecolor='white');plt.show();plt.close(fig)
def digest(p):return hashlib.sha256(Path(p).read_bytes()).hexdigest()
def metrics(y,p):return {'n':int(len(y)),'MAE':float(mean_absolute_error(y,p)),'RMSE':float(np.sqrt(mean_squared_error(y,p)))}
def candidate(name):
 if name=='median':return DummyRegressor(strategy='median')
 return make_pipeline(StandardScaler(),Ridge(alpha=float(name.split('-')[1]),solver='svd'))
print('Actual environment:',ENV);print('Declared configuration:',CONFIG)
responses={};observations={};challenge_observations={}
from IPython.display import display,FileLink


## Task 1: Recover observations by ID (7 min)

**Purpose:** Distinguish row membership, row order and a random seed.

**Do:** Use a 20-ID paper-sized example. A seeded permutation saves 15 training IDs. Reverse the source array and reuse positions: identify the changed membership. Recover rows using the SAVED ID sequence, not just a set filter. No model or 442-row dataset is needed yet.

**Stopping check:** Membership is recovered and the final ID sequence is identical. A seed alone does not bind positions to observations.

Fill the prediction before running, then explain one actual output and its boundary.

In [ ]:
responses["task1_prediction"] = ''

In [ ]:
toy_ids=np.array([f'row-{i:02d}' for i in range(20)])
positions=np.random.default_rng(2026).permutation(20)[:15]
saved_ids=toy_ids[positions].tolist(); reversed_ids=toy_ids[::-1]
wrong_ids=reversed_ids[positions].tolist(); lookup={row_id:row_id for row_id in reversed_ids}
recovered=[lookup[row_id] for row_id in saved_ids]
assert recovered==saved_ids
observations['task1']={'saved':saved_ids,'wrong':wrong_ids,'recovered':recovered,
    'positional_membership_changed':set(wrong_ids)!=set(saved_ids),'same_recovered_sequence':recovered==saved_ids}
print(json.dumps(observations['task1'],indent=2))

In [ ]:
responses["task1_explanation"] = ''

## Task 2: Freeze raw data and check preprocessing scope (8 min)

**Purpose:** Make fitted preprocessing belong only to training rows.

**Do:** Load the retained 442-row packaged regression fixture with scaled=False. Create source IDs before splitting; freeze CSV and 265/88/89 membership. Check disjointness and the full union. Fit a diagnostic scaler on training rows ONLY, then compare its means with training means. The all-row mean gap is descriptive, not a second competing leaky model.

**Stopping check:** 442 unique IDs; 265/88/89 disjoint exhaustive membership; training means match the diagnostic scaler. No clinical interpretation is required.

Fill the prediction before running, then explain one actual output and its boundary.

In [ ]:
responses["task2_prediction"] = ''

In [ ]:
data=load_diabetes(scaled=False)
X=np.asarray(data.data,dtype=float);y=np.asarray(data.target,dtype=float)
FEATURES=list(data.feature_names);IDS=np.array([f'diabetes-{i:04d}' for i in range(len(y))])
assert X.shape==(442,10) and y.shape==(442,)
assert np.isfinite(X).all() and np.isfinite(y).all()
order=np.random.default_rng(CONFIG['seed']).permutation(len(y))
train_idx=np.sort(order[:265]);val_idx=np.sort(order[265:353]);test_idx=np.sort(order[353:])
SPLITS={k:IDS[idx].tolist() for k,idx in [('train',train_idx),('validation',val_idx),('test',test_idx)]}
assert all(not(set(SPLITS[a])&set(SPLITS[b])) for a,b in [('train','validation'),('train','test'),('validation','test')])
assert set(sum(SPLITS.values(),[]))==set(IDS)
Xtr,ytr=X[train_idx],y[train_idx];Xva,yva=X[val_idx],y[val_idx];Xte,yte=X[test_idx],y[test_idx]
HEADER=['row_id',*FEATURES,'target']
DATA_PATH=BUNDLE_DIR/'frozen.csv'
with DATA_PATH.open('w',newline='',encoding='utf-8') as f:
 w=csv.writer(f);w.writerow(HEADER)
 for i in range(len(y)):w.writerow([IDS[i],*[repr(float(v)) for v in X[i]],repr(float(y[i]))])
(BUNDLE_DIR/'split.json').write_text(json.dumps(SPLITS,indent=2),encoding='utf-8')
data_sha256=digest(DATA_PATH)
print('Rows / features:',X.shape,'Columns:',HEADER)
print('Partition sizes:',{k:len(v) for k,v in SPLITS.items()});print('CSV SHA-256:',data_sha256)

# Show training targets only; final-test outcomes are first displayed in Task 5.
fig,ax=plt.subplots(layout='constrained');ax.hist(ytr,bins=14,color=TEAL)
ax.set(xlabel='Training target only',ylabel='Count',title='Training cohort; validation/test outcomes not displayed here')
finish(fig,'w10-target-distributions.png')

scope_scaler=StandardScaler().fit(Xtr)
np.testing.assert_allclose(scope_scaler.mean_,Xtr.mean(axis=0),atol=1e-12)
mean_gap=(X.mean(axis=0)-Xtr.mean(axis=0))/scope_scaler.scale_
observations['task2']={'counts':{k:len(v) for k,v in SPLITS.items()},'training_mean_matches':True,
 'mean_gap_train_sd':mean_gap.tolist(),'row_id_is_feature':False}
fig,ax=plt.subplots(layout='constrained');ax.bar(FEATURES,mean_gap,color=RED);ax.axhline(0,color='black')
ax.set(xlabel='Input coordinate',ylabel='(All-row mean - train mean) / train SD');finish(fig,'w10-preprocessing-scope.png')
print(observations['task2'])

In [ ]:
responses["task2_explanation"] = ''

## Task 3: Fit candidates and freeze the validation choice (15 min)

**Purpose:** Separate training loss, model selection and final evaluation.

**Do:** Fit the training-target median and Ridge alpha 1, 20, 100 pipelines using the SAME training rows. Ridge minimizes squared error plus an L2 penalty; selection uses validation MAE. The listed order breaks exact ties. Save the choice now; final-test outcomes appear only in Task 5. Inspect the actual coefficient-norm curve separately from validation MAE.

**Stopping check:** Four training-only candidates; one saved validation choice; scaler means agree with Task 2. The median baseline remains eligible to win.

Fill the prediction before running, then explain one actual output and its boundary.

In [ ]:
responses["task3_prediction"] = ''

In [ ]:
models={name:candidate(name).fit(Xtr,ytr) for name in CONFIG['candidate_order']}
validation={name:metrics(yva,m.predict(Xva)) for name,m in models.items()}
selected=min(CONFIG['candidate_order'],key=lambda name:(validation[name]['MAE'],CONFIG['candidate_order'].index(name)))
print('Validation:',json.dumps(validation,indent=2));print('Selected before final test:',selected)
fig,ax=plt.subplots(layout='constrained')
ax.bar(range(4),[validation[n]['MAE'] for n in models],color=[TEAL if n==selected else RED for n in models])
ax.set_xticks(range(4),['Median','Ridge 1','Ridge 20','Ridge 100']);ax.set_ylabel('Validation MAE (target units)');finish(fig,'w10-validation-selection.png')
fig,ax=plt.subplots(layout='constrained');alphas=CONFIG['alphas']
norms=[float(np.linalg.norm(models['ridge-'+f'{a:g}'].named_steps['ridge'].coef_)) for a in alphas]
ax.plot(alphas,norms,'o-',color=TEAL,lw=2);ax.set_xscale('log');ax.set(xlabel='Ridge alpha',ylabel='Standardized coefficient L2 norm');finish(fig,'w10-regularization-path.png')
for name,m in models.items():
 if name!='median':np.testing.assert_allclose(m.named_steps['standardscaler'].mean_,scope_scaler.mean_)
selected_before_test=selected
observations['task3']={'validation':validation,'selected':selected,'test_used_for_selection':False}

In [ ]:
responses["task3_explanation"] = ''

## Review the portable replay program

Expand the code cell below and review its named steps: verify hashes → read the data contract → reconstruct saved ID cohorts → read reference predictions → refit the frozen candidate → compare aligned vectors. This cell writes the provided source and imports its functions; it does not train a model. Task 4 calls only the schema reader. Task 5 executes a copied file in a separate process. Only run reviewed, trusted code; a checksum is not a signature.

In [ ]:
RUNNER = r'''
"""Reviewed Week10 replay program. Frozen files in, aligned comparison out."""
import argparse,csv,json,hashlib,os,platform
from pathlib import Path
import numpy as np
import sklearn
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from sklearn.dummy import DummyRegressor
PAYLOAD={'frozen.csv','split.json','config.json','reference-predictions.csv','replay.py','result.json'}
def digest(path):return hashlib.sha256(Path(path).read_bytes()).hexdigest()
def read_contract(root,config):
    root=Path(root)
    with (root/'frozen.csv').open(newline='',encoding='utf-8') as f:
        reader=csv.DictReader(f)
        if reader.fieldnames!=['row_id',*config['features'],'target']:raise ValueError('schema: exact column order required')
        rows=list(reader)
    if len(rows)!=config['row_count']:raise ValueError('schema: row count')
    if any(None in r or any(v is None for v in r.values()) for r in rows):raise ValueError('schema: row width')
    ids=[r['row_id'] for r in rows]
    if any(not i.strip() for i in ids):raise ValueError('schema: empty row ID')
    if len(set(ids))!=len(ids):raise ValueError('schema: duplicate row ID')
    values=np.array([[float(r[k]) for k in [*config['features'],'target']] for r in rows])
    if not np.isfinite(values).all():raise ValueError('schema: non-finite numeric value')
    splits=json.loads((root/'split.json').read_text())
    if set(splits)!={'train','validation','test'}:raise ValueError('schema: split names')
    if not all(isinstance(v,list) and all(isinstance(i,str) for i in v) for v in splits.values()):raise ValueError('schema: ID lists required')
    flat=sum(splits.values(),[])
    if len(flat)!=len(set(flat)) or set(flat)!=set(ids):raise ValueError('schema: split overlap or missing IDs')
    if {k:len(v) for k,v in splits.items()}!=config['split_counts']:raise ValueError('schema: split counts')
    lookup={r['row_id']:r for r in rows}
    def cohort(name):
        subset=[lookup[i] for i in splits[name]]
        return np.array([[float(r[k]) for k in config['features']] for r in subset]),np.array([float(r['target']) for r in subset])
    return cohort,splits

def run(root,out):
    root=Path(root).resolve();manifest=json.loads((root/'manifest.json').read_text())
    if manifest.get('format')!='MLES-W10-v2' or set(manifest['sha256'])!=PAYLOAD:raise ValueError('manifest: exact required file list and format')
    for name,value in manifest['sha256'].items():
        if digest(root/name)!=value:raise ValueError('checksum mismatch: '+name)
    config=json.loads((root/'config.json').read_text());cohort,splits=read_contract(root,config)
    Xtr,ytr=cohort('train');Xte,yte=cohort('test')
    chosen=config['selected'];allowed=['median','ridge-1','ridge-20','ridge-100']
    if config['candidate_order']!=allowed or chosen not in allowed:raise ValueError('config: undeclared candidate')
    # The runner itself reads and validates reference predictions; no notebook globals are needed.
    with (root/'reference-predictions.csv').open(newline='',encoding='utf-8') as f:
        reader=csv.DictReader(f)
        if reader.fieldnames!=['row_id','observed','predicted']:raise ValueError('reference: exact header required')
        reference=list(reader)
    if any(None in r or any(v is None for v in r.values()) for r in reference):raise ValueError('reference: row width')
    if [r['row_id'] for r in reference]!=splits['test']:raise ValueError('reference: test ID sequence mismatch')
    ref_y=np.array([float(r['observed']) for r in reference]);ref_p=np.array([float(r['predicted']) for r in reference])
    if not np.isfinite(ref_y).all() or not np.isfinite(ref_p).all():raise ValueError('reference: non-finite value')
    np.testing.assert_array_equal(ref_y,yte)
    model=DummyRegressor(strategy='median') if chosen=='median' else make_pipeline(StandardScaler(),Ridge(alpha=float(chosen.split('-')[1]),solver='svd'))
    model.fit(Xtr,ytr);pred=model.predict(Xte)
    np.testing.assert_allclose(pred,ref_p,atol=config['atol'],rtol=config['rtol'])
    error=pred-yte;delta=pred-ref_p
    result={'selected':chosen,'data_sha256':digest(root/'frozen.csv'),'test_ids':splits['test'],'predictions':pred.tolist(),
        'metrics':{'n':len(yte),'MAE':float(np.abs(error).mean()),'RMSE':float(np.sqrt((error**2).mean()))},
        'reference_comparison_passed':True,'max_reference_difference':float(np.max(np.abs(delta))),
        'process_id':os.getpid(),'working_directory_name':Path.cwd().name,
        'environment':{'python':platform.python_version(),'numpy':np.__version__,'scikit_learn':sklearn.__version__},
        'scope':'Fresh interpreter and working directory, same installed environment; not independent scientific replication'}
    Path(out).write_text(json.dumps(result,indent=2,allow_nan=False),encoding='utf-8')
    print(json.dumps({'selected':chosen,'test_n':len(yte),'reference_comparison_passed':True}))
    return result
if __name__=='__main__':
    p=argparse.ArgumentParser();p.add_argument('--root',required=True);p.add_argument('--out',required=True);a=p.parse_args();run(a.root,a.out)

'''
(BUNDLE_DIR/'replay.py').write_text(RUNNER,encoding='utf-8')
import importlib.util
spec=importlib.util.spec_from_file_location('reviewed_w10_runner',BUNDLE_DIR/'replay.py')
runner_module=importlib.util.module_from_spec(spec);spec.loader.exec_module(runner_module)
print('Reviewed replay source written; no fitting performed by this cell.')

## Task 4: Reject malformed copies before fitting (12 min)

**Purpose:** Give the data interface executable stopping conditions.

**Do:** Expand and review the provided replay program before this task. Work on temporary copies. Predict the expected rejection for duplicate IDs, NaN, wrong headers, row width, empty IDs and an invalid partition. Call the schema reader directly, so this test checks input contracts without training. Keep the original CSV hash unchanged.

**Stopping check:** Every declared bad copy is rejected with a recorded reason. The original reference bytes remain unchanged.

Fill the prediction before running, then explain one actual output and its boundary.

In [ ]:
responses["task4_prediction"] = ''

In [ ]:
FROZEN_CONFIG={**CONFIG,'features':FEATURES,'row_count':442,'split_counts':{'train':265,'validation':88,'test':89},'selected':selected,'environment':ENV}
def contract_probe(kind):
 with tempfile.TemporaryDirectory(prefix='mles-w10-probe-') as work:
  target=Path(work);shutil.copy2(DATA_PATH,target/'frozen.csv');shutil.copy2(BUNDLE_DIR/'split.json',target/'split.json')
  with (target/'frozen.csv').open(newline='',encoding='utf-8') as f:rows=list(csv.reader(f))
  if kind=='duplicate ID':rows[2][0]=rows[1][0]
  elif kind=='non-finite':rows[1][1]='nan'
  elif kind=='column order':rows=[[r[1],r[0],*r[2:]] for r in rows]
  elif kind=='extra value':rows[1].append('unexpected')
  elif kind=='missing value':rows[1].pop()
  elif kind=='empty ID':rows[1][0]=''
  elif kind=='invalid partition':
   bad=json.loads((target/'split.json').read_text());bad['test'][0]=bad['train'][0];(target/'split.json').write_text(json.dumps(bad))
  else:raise ValueError(kind)
  with (target/'frozen.csv').open('w',newline='',encoding='utf-8') as f:csv.writer(f).writerows(rows)
  try:runner_module.read_contract(target,FROZEN_CONFIG)
  except ValueError as error:return {'case':kind,'rejected':True,'reason':str(error)}
  raise AssertionError('Malformed copy unexpectedly accepted: '+kind)
contract_checks=[contract_probe(k) for k in ['duplicate ID','non-finite','column order','extra value','missing value','empty ID','invalid partition']]
assert digest(DATA_PATH)==data_sha256
observations['task4']=contract_checks
print(json.dumps(contract_checks,indent=2))
fig,ax=plt.subplots(layout='constrained');ax.barh([r['case'] for r in contract_checks],[1]*len(contract_checks),color=TEAL)
ax.set(xlim=(0,1.2),xlabel='Actual rejection outcome');ax.set_xticks([0,1],['Accepted','Rejected']);finish(fig,'w10-contract-tests.png')

In [ ]:
responses["task4_explanation"] = ''

## Task 5: Evaluate once and replay from copied files (18 min)

**Purpose:** Build a handover that can check its own predictions without notebook variables.

**Do:** Open final-test outcomes after the validation choice is frozen. Compare the selected model with the declared median baseline; inspect one actual failure ID. Build six payload files and a manifest. Copy the bundle into a new temporary directory and invoke its replay.py in a NEW Python interpreter. The runner must read the reference-predictions CSV itself and compare all 89 aligned values. Record a different process ID; do not call the runner function in this notebook and call that a fresh process.

**Stopping check:** All file hashes, schema and exact test-ID sequence pass. The child independently compares predictions within atol=1e-8 and rtol=1e-10. This still shares installed dependencies.

Fill the prediction before running, then explain one actual output and its boundary.

In [ ]:
responses["task5_prediction"] = ''

In [ ]:
test_prediction={n:models[n].predict(Xte) for n in dict.fromkeys(['median',selected])}
test_metrics={n:metrics(yte,p) for n,p in test_prediction.items()}
pred=test_prediction[selected];residual=pred-yte
print('Final test:',json.dumps(test_metrics,indent=2))
fig,ax=plt.subplots(layout='constrained');names=list(test_metrics);width=.34;xx=np.arange(len(names))
for j,m in enumerate(['MAE','RMSE']):ax.bar(xx+(j-.5)*width,[test_metrics[n][m] for n in names],width,label=m)
ax.set_xticks(xx,names);ax.set_ylabel('Target units');ax.legend();finish(fig,'w10-test-metrics.png')
fig,ax=plt.subplots(layout='constrained');ax.scatter(yte,pred,s=36,alpha=.8,color=TEAL)
limits=[float(min(yte.min(),pred.min())),float(max(yte.max(),pred.max()))];ax.plot(limits,limits,'k--',lw=1)
ax.set(xlabel='Observed test target',ylabel='Selected-model prediction');finish(fig,'w10-test-parity.png')
fig,ax=plt.subplots(layout='constrained');ax.scatter(pred,residual,s=36,color=RED,alpha=.8);ax.axhline(0,color='black',lw=1)
ax.set(xlabel='Selected-model prediction',ylabel='Prediction - observation');finish(fig,'w10-test-residuals.png')
fig,ax=plt.subplots(layout='constrained')
for n,p in test_prediction.items():
 e=np.sort(np.abs(p-yte));ax.plot(e,np.arange(1,len(e)+1)/len(e),label=n,lw=2)
ax.set(xlabel='Absolute error (target units)',ylabel='Empirical cumulative fraction');ax.legend();finish(fig,'w10-error-cdf.png')
worst=np.argsort(np.abs(residual))[-5:][::-1]
worst_rows=[{'row_id':IDS[test_idx[j]],'observed':float(yte[j]),'predicted':float(pred[j]),'residual':float(residual[j])} for j in worst]
print('Five largest absolute residuals:',json.dumps(worst_rows,indent=2))
assert selected==selected_before_test
FROZEN_CONFIG={**CONFIG,'features':FEATURES,'row_count':442,'split_counts':{'train':265,'validation':88,'test':89},'selected':selected,'environment':ENV}
(BUNDLE_DIR/'config.json').write_text(json.dumps(FROZEN_CONFIG,indent=2),encoding='utf-8')
with (BUNDLE_DIR/'reference-predictions.csv').open('w',newline='',encoding='utf-8') as f:
 w=csv.writer(f);w.writerow(['row_id','observed','predicted'])
 for i,actual,p in zip(IDS[test_idx],yte,pred):w.writerow([i,repr(float(actual)),repr(float(p))])
record={'week':10,'data_scope':'Packaged regression dataset; statistical teaching only, not clinical validation',
 'config':CONFIG,'environment':ENV,'data_sha256':data_sha256,'selected_on_validation':selected,
 'validation':validation,'test':test_metrics,'worst_test_rows':worst_rows,
 'claim_limit':'Frozen random-row teaching split; no external clinical, causal or deployment claim.'}
(BUNDLE_DIR/'result.json').write_text(json.dumps(record,indent=2,allow_nan=False),encoding='utf-8')
BUNDLE_FILES=['frozen.csv','split.json','config.json','reference-predictions.csv','replay.py','result.json']
manifest={'format':'MLES-W10-v2','sha256':{name:digest(BUNDLE_DIR/name) for name in BUNDLE_FILES}}
(BUNDLE_DIR/'manifest.json').write_text(json.dumps(manifest,indent=2),encoding='utf-8')
print('Handover files:',BUNDLE_FILES+['manifest.json'])
def fresh_process(root,label):
 output=(ARTIFACTS/(label+'-process-result.json')).resolve()
 with tempfile.TemporaryDirectory(prefix='mles-w10-fresh-') as work:
  copied=Path(work)/'handover';shutil.copytree(root,copied)
  proc=subprocess.run([sys.executable,'-I',str(copied/'replay.py'),'--root',str(copied),'--out',str(output)],cwd=work,capture_output=True,text=True,timeout=120)
  if proc.returncode:raise RuntimeError(proc.stderr)
  result=json.loads(output.read_text());assert result['process_id']!=os.getpid()
  assert result['working_directory_name']==Path(work).name
  return result
replay=fresh_process(BUNDLE_DIR,'reference')
assert replay['test_ids']==SPLITS['test'] and replay['reference_comparison_passed']
np.testing.assert_allclose(replay['predictions'],pred,atol=CONFIG['atol'],rtol=CONFIG['rtol'])
replay_diff=np.asarray(replay['predictions'])-pred;max_replay_difference=float(np.max(np.abs(replay_diff)))
observations['task5']={'different_process':True,'same_installed_environment':True,'runner_compares_its_own_reference':True,'max_difference':max_replay_difference,'worst_case':worst_rows[0]}
fig,ax=plt.subplots(layout='constrained');ax.scatter(np.arange(len(pred)),replay_diff,color=TEAL);ax.axhline(0,color='black');ax.set(ylim=(-1e-7,1e-7),xlabel='Saved test-ID position',ylabel='Fresh process - reference prediction');finish(fig,'w10-replay-differences.png')
print(observations['task5'])

In [ ]:
responses["task5_explanation"] = ''

## Task 6: Reorder CSV rows, preserve the records (15 min)

**Purpose:** Distinguish changed bytes from changed records, then test row-order independence.

**Do:** Reverse data rows in a temporary COPY, keep the header and saved split-ID lists. Confirm that the exact CSV hash changes while the entire ID-keyed record mapping stays equal. Record this reviewed transformation and update ONLY that copy’s CSV digest in its manifest. Run a fresh process and compare aligned predictions. The reference bundle stays unchanged. Updating a manifest alone never proves a transformation is legitimate.

**Stopping check:** Different CSV hashes, equal ID-keyed values, equal test-ID order and predictions within tolerance. No new winner is selected.

Fill the prediction before running, then explain one actual output and its boundary.

In [ ]:
responses["task6_prediction"] = ''

In [ ]:
with tempfile.TemporaryDirectory(prefix='mles-w10-reorder-') as work:
 copied=Path(work)/'handover';shutil.copytree(BUNDLE_DIR,copied)
 with (copied/'frozen.csv').open(newline='',encoding='utf-8') as f:rows=list(csv.reader(f))
 original_records={row[0]:row[1:] for row in rows[1:]}
 reordered=[rows[0],*rows[:0:-1]]
 assert {row[0]:row[1:] for row in reordered[1:]}==original_records
 with (copied/'frozen.csv').open('w',newline='',encoding='utf-8') as f:csv.writer(f).writerows(reordered)
 new_hash=digest(copied/'frozen.csv');assert new_hash!=data_sha256
 changed_manifest=json.loads((copied/'manifest.json').read_text());changed_manifest['sha256']['frozen.csv']=new_hash
 (copied/'manifest.json').write_text(json.dumps(changed_manifest,indent=2))
 reordered_replay=fresh_process(copied,'reordered')
 assert reordered_replay['test_ids']==SPLITS['test']
 np.testing.assert_allclose(reordered_replay['predictions'],pred,atol=CONFIG['atol'],rtol=CONFIG['rtol'])
observations['task6']={'reference_sha256':data_sha256,'reviewed_copy_sha256':new_hash,'ID_keyed_records_equal':True,'saved_ID_sequence_equal':True,'max_prediction_difference':reordered_replay['max_reference_difference'],'reference_unchanged':digest(DATA_PATH)==data_sha256}
assert observations['task6']['reference_unchanged'];print(observations['task6'])

In [ ]:
responses["task6_explanation"] = ''

## Task 7: Check a coordinate explanation (12 min)

**Purpose:** Explain the fitted function in its declared coordinate system.

**Do:** If the selected candidate is Ridge, inspect it; if the median won, explicitly use the separately declared Ridge-20 interpretation candidate. Recover raw coefficients and intercept, then check all predictions. Increase coordinate bmi by 0.1 TRAINING SD, holding others fixed; compare every prediction delta with 0.1 times its standardized coefficient. This tests a function, not a feasible medical intervention.

**Stopping check:** The raw equation and the entire perturbation vector match within tolerance; the interpretation candidate is recorded.

Fill the prediction before running, then explain one actual output and its boundary.

In [ ]:
responses["task7_prediction"] = ''

In [ ]:
explain_name=selected if selected!='median' else 'ridge-20'
explain_model=models[explain_name];scaler=explain_model.named_steps['standardscaler'];reg=explain_model.named_steps['ridge']
raw_coef=reg.coef_/scaler.scale_;raw_intercept=float(reg.intercept_-np.dot(raw_coef,scaler.mean_))
np.testing.assert_allclose(raw_intercept+Xte@raw_coef,explain_model.predict(Xte),atol=1e-8,rtol=1e-10)
coefficient_record={'candidate':explain_name,'features':FEATURES,'standardized':reg.coef_.tolist(),
 'raw':raw_coef.tolist(),'raw_intercept':raw_intercept,'training_mean':scaler.mean_.tolist(),'training_scale':scaler.scale_.tolist()}
fig,ax=plt.subplots(layout='constrained');ax.barh(FEATURES,reg.coef_,color=[TEAL if v>=0 else RED for v in reg.coef_]);ax.axvline(0,color='black',lw=1)
ax.set(xlabel='Response change per +1 training SD',ylabel='Input coordinate',title=explain_name+' / function, not causation');finish(fig,'w10-coefficients.png')
mean_gap=(X.mean(axis=0)-Xtr.mean(axis=0))/Xtr.std(axis=0)
fig,ax=plt.subplots(layout='constrained');ax.bar(FEATURES,mean_gap,color=RED);ax.axhline(0,color='black',lw=1)
ax.set(xlabel='Feature',ylabel='(All-row mean - train mean) / train SD');finish(fig,'w10-preprocessing-scope.png')
j=FEATURES.index('bmi');changed=Xte.copy();changed[:,j]+=.1*scaler.scale_[j]
observed_delta=explain_model.predict(changed)-explain_model.predict(Xte);expected_delta=.1*reg.coef_[j]
np.testing.assert_allclose(observed_delta,np.full(len(yte),expected_delta),atol=CONFIG['atol'],rtol=CONFIG['rtol'])
observations['task7']={'candidate':explain_name,'coordinate':FEATURES[j],'step_training_SD':.1,'expected_delta':float(expected_delta),'max_disagreement':float(np.max(np.abs(observed_delta-expected_delta)))}
print(observations['task7'])

In [ ]:
responses["task7_explanation"] = ''

## Task 8: Audit, explain and hand over (13 min)

**Purpose:** Connect a portable artifact with an actual result and a bounded claim.

**Do:** Verify all six payload hashes again. Show a partner the validation winner, one actual failure row, test MAE and the separate replay difference. Locate the CSV, split, config, reviewed source, reference predictions and result record. Complete the three reflection fields and export below. Keep the eight predictions and explanations tied to your own observed outputs.

**Stopping check:** A partner can locate the evidence without your hidden variables. All 19 fields are checked for completeness; blank fields remain an honest draft.

Fill the prediction before running, then explain one actual output and its boundary.

In [ ]:
responses["task8_prediction"] = ''

In [ ]:
checks={name:digest(BUNDLE_DIR/name)==sha for name,sha in manifest['sha256'].items()}
assert all(checks.values()) and digest(DATA_PATH)==data_sha256
observations['task8']={'payload_hashes':checks,'selected':selected,'test_metric':test_metrics[selected],'one_failure_ID':worst_rows[0]['row_id'],'fresh_process_difference':max_replay_difference}
print(json.dumps(observations['task8'],indent=2))

In [ ]:
responses["task8_explanation"] = ''

## Optional A: tamper detection; Optional B: explicit name adapter

Complete all eight core tasks first. Set `RUN_CHALLENGES=True` below only if time remains. A changes a copied CSV without updating its manifest and expects rejection before an output is written. B restores the canonical feature order by name and verifies exact values. Both leave the reference unchanged. Neither checksum matching nor a working adapter establishes scientific validity.

In [ ]:

RUN_CHALLENGES=False
challenge_observations={}
if RUN_CHALLENGES:
 # A: unchanged manifest must reject a changed file, even a blank-line-only byte change.
 with tempfile.TemporaryDirectory(prefix='mles-w10-tamper-') as work:
  copied=Path(work)/'handover';shutil.copytree(BUNDLE_DIR,copied)
  (copied/'frozen.csv').write_bytes((copied/'frozen.csv').read_bytes()+b'\n')
  output=Path(work)/'must-not-exist.json'
  proc=subprocess.run([sys.executable,'-I',str(copied/'replay.py'),'--root',str(copied),'--out',str(output)],cwd=work,capture_output=True,text=True,timeout=120)
  assert proc.returncode!=0 and 'checksum mismatch' in proc.stderr and not output.exists()
  challenge_observations['tamper']={'changed_bytes_rejected':True,'no_result_written':True}
 # B: explicit name adapter; do not silently accept a positional mismatch.
 order=list(reversed(FEATURES));wrong=X[:,[FEATURES.index(name) for name in order]]
 restored=np.column_stack([wrong[:,order.index(name)] for name in FEATURES])
 np.testing.assert_array_equal(restored,X)
 challenge_observations['name_adapter']={'strict_rejection':contract_probe('column order')['rejected'],'canonical_values_restored':True}
 assert digest(DATA_PATH)==data_sha256
 print(challenge_observations)
else:print('Optional branches skipped. Eight core tasks are sufficient.')


## Reflect, peer-review and download

Use the remaining 20 studio minutes to show the saved winner, one actual failure ID, the prediction error against outcomes, and the separate replay difference. Fill the three fields below. Eight predictions + eight explanations + three reflections = **19 fields**. Blanks produce a draft; filled fields only indicate readiness for discussion. Write an accurate AI-use disclosure (`none` if appropriate).

In [ ]:
responses['controlled_change'] = ''
responses['limitation'] = ''
responses['ai_disclosure'] = ''

In [ ]:

keys=[f'task{i}_{kind}' for i in range(1,9) for kind in ['prediction','explanation']]+['controlled_change','limitation','ai_disclosure']
blank=[k for k in keys if not isinstance(responses.get(k),str) or not responses[k].strip()]
status='draft' if blank else 'ready for peer discussion; completeness only, not correctness'
record.update({'replay_max_prediction_difference':max_replay_difference,'contract_tests':contract_checks,'replay_scope':replay['scope'],'observations':observations,'responses':responses,'blank_fields':blank,'response_status':status,'optional':challenge_observations,'coefficients':coefficient_record})
(ARTIFACTS/'week10-evidence.json').write_text(json.dumps(record,indent=2,allow_nan=False))
HANDOVER_ZIP=ARTIFACTS/'week10-workflow-handover.zip'
with zipfile.ZipFile(HANDOVER_ZIP,'w',zipfile.ZIP_DEFLATED) as z:
 for name in sorted([*BUNDLE_FILES,'manifest.json']):z.write(BUNDLE_DIR/name,arcname='handover/'+name)
 for p in sorted(ARTIFACTS.glob('w10-*.png')):z.write(p,arcname='figures/'+p.name)
 z.write(ARTIFACTS/'week10-evidence.json',arcname='week10-evidence.json')
with zipfile.ZipFile(HANDOVER_ZIP) as z:assert z.testzip() is None
print(status,'Completed fields:',19-len(blank),'/19','Missing:',blank)
print('Download ZIP:',HANDOVER_ZIP,'Save your edited notebook too.')
print('After extraction and source review: python handover/replay.py --root handover --out replay-result.json')
display(FileLink(str(HANDOVER_ZIP)))


## Summary and examination bridge

Preserve observation identity and declared choices. Make schema rejection explicit. Give the replay program enough files to compare its own reference predictions. Distinguish exact bytes from records, and distinguish replay difference from prediction error. Hand over evidence with a bounded conclusion.

**Written-exam practice:** explain why a seed alone is insufficient and compute the tolerance for a reference value of 10. **Oral-exam practice:** use a real row from your run to explain the selected model, one failure and one limitation. This practice does not replace either examination.

## Primary references

- [Raw packaged regression data](https://scikit-learn.org/stable/modules/generated/sklearn.datasets.load_diabetes.html)
- [Training-only pipelines](https://scikit-learn.org/stable/common_pitfalls.html)
- [Random stream compatibility](https://numpy.org/doc/stable/reference/random/compatibility.html)
- [Model persistence and trusted code](https://scikit-learn.org/stable/model_persistence.html)

Documentation checked 27 October 2026. No new weekly graded Homework is introduced.